# Assignment 1 — Building a Vision Model with Keras

**U of T DSI — Deep Learning**

This notebook completes the five tasks in the course's Assignment 1: data inspection, a linear baseline, a convolutional neural network (CNN), controlled experiments, and final model evaluation.

**Execution note:** Run the notebook from top to bottom using your Python 3.11 `.venv311` environment. The code calculates test metrics when executed; no training accuracy or experiment result has been invented. Fashion-MNIST downloads the first time it is used.

**Experiment design:** To keep CPU-based experiments manageable, hyperparameter trials use the *same fixed subset* of the training data, with the *same number of epochs* for a fair comparison. Baseline, reference CNN, and final model are trained on the full training set. The final choice is made using validation accuracy; test-set figures are also reported to match the assignment instructions. Selecting only on validation avoids choosing a model by looking at its test scores.

---

# 1. Loading and Inspecting the Dataset

Fashion-MNIST is a ten-class image classification dataset containing grayscale, 28×28-pixel clothing images. The tasks are to inspect shapes, one-hot encode labels, and visualize examples.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical

# Reproducible random initialization and sample selection.
tf.keras.utils.set_random_seed(42)

(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("TensorFlow version:", tf.__version__)
print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)
print("Image intensity range:", float(X_train.min()), "to", float(X_train.max()))
print("Unique classes:", np.unique(y_train))


In [ ]:
# Labels are 0–9 integers, so use one-hot encoded targets for categorical crossentropy.
y_train_onehot = to_categorical(y_train, num_classes=10)
y_test_onehot = to_categorical(y_test, num_classes=10)

print("One-hot training labels:", y_train_onehot.shape)
print("First label:", y_train[0])
print("Its one-hot encoding:", y_train_onehot[0])

assert X_train.shape == (60000, 28, 28)
assert X_test.shape == (10000, 28, 28)
assert y_train_onehot.shape == (60000, 10)
assert y_test_onehot.shape == (10000, 10)


In [ ]:
# Visualize nine examples from the training set.
rng = np.random.default_rng(42)
example_indices = rng.choice(len(X_train), size=9, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax, idx in zip(axes.flat, example_indices):
    ax.imshow(X_train[idx], cmap="gray")
    ax.set_title(f"{class_names[y_train[idx]]} (class {y_train[idx]})", fontsize=9)
    ax.axis("off")
plt.suptitle("Fashion-MNIST sample images")
plt.tight_layout()
plt.show()


### Reflection — Dataset quality

The dataset has the expected format: each image is a normalized, single-channel 28×28 picture, and the labels represent ten clothing categories. The resolution is low, so fine details such as fabric texture and shape are difficult to see. Some categories, especially **shirt**, **T-shirt/top**, **coat**, and **pullover**, have similar silhouettes. This may cause classification mistakes even with a well-trained model. Normalizing pixel intensities to the range 0–1 helps optimization, but it does not restore image detail that was not recorded. The displayed images and class labels provide a useful initial check of the input data.


# 2. Baseline Model

The assignment asks for a simple linear model with **Flatten → Dense(10)** and *no activation in the output layer*. These ten outputs are **logits**, not probabilities. Therefore `CategoricalCrossentropy(from_logits=True)` is the appropriate classification loss. This respects the requested linear architecture while making training suitable for the ten classes.


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy

# Linear model: Flatten plus one Dense layer without an activation function.
baseline_model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),
    Dense(10),
], name="linear_baseline")

baseline_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

baseline_model.summary()
baseline_history = baseline_model.fit(
    X_train, y_train_onehot,
    epochs=4, batch_size=128, validation_split=0.1,
    verbose=2
)
baseline_loss, baseline_accuracy = baseline_model.evaluate(
    X_test, y_test_onehot, verbose=0
)
print(f"Baseline test loss: {baseline_loss:.4f}")
print(f"Baseline test accuracy: {baseline_accuracy:.2%}")


### Reflection — Baseline performance

The baseline's **measured test loss and accuracy** are printed in the previous cell. Its architecture is intentionally simple: after flattening, a linear transformation scores each of the ten clothing classes. It cannot explicitly learn local patterns such as edges or shapes that appear in different parts of an image. I therefore expect this baseline to provide a useful reference but to be less effective than a CNN on many fashion categories. The numerical result above, rather than an assumed percentage, is the basis for judging its performance.


# 3. Building and Evaluating a Simple CNN Model

The CNN uses two convolution-and-pooling stages to learn local visual patterns, followed by a dense classification layer. A 10-class softmax output is trained with categorical crossentropy and evaluated on held-out test images.


In [ ]:
# CNNs expect an explicit channel dimension (grayscale = one channel).
X_train_cnn = X_train[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]
assert X_train_cnn.shape == (60000, 28, 28, 1)

cnn_model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
], name="reference_cnn")

cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
cnn_model.summary()
cnn_history = cnn_model.fit(
    X_train_cnn, y_train_onehot,
    epochs=4, batch_size=128, validation_split=0.1,
    verbose=2
)
cnn_loss, cnn_accuracy = cnn_model.evaluate(
    X_test_cnn, y_test_onehot, verbose=0
)
print(f"Reference CNN test loss: {cnn_loss:.4f}")
print(f"Reference CNN test accuracy: {cnn_accuracy:.2%}")
print(f"CNN minus baseline accuracy: {(cnn_accuracy - baseline_accuracy) * 100:+.2f} percentage points")


### Reflection — CNN versus baseline

The code above reports both models' test accuracy and the **difference in percentage points**. The CNN's convolution layers can recognize local patterns and reuse the same filters across an image; the baseline loses this spatial structure after flattening. If the measured difference is positive, that is a plausible explanation for the CNN's improvement. If it is small or negative, factors such as training duration, random initialization, and hyperparameters should also be considered. The printed comparison supplies the actual answer rather than an unverified score.


# 4. Designing and Running Controlled Experiments

**Hyperparameter:** number of filters in the first convolution layer: **16, 32, 64** (the second layer uses twice as many filters).

**Regularization:** `Dropout(0.25)` versus no dropout.

All trials use exactly the same 15,000 randomly chosen training examples, the same validation split, optimizer, batch size, and 3 training epochs. Each trial creates **a new model** to avoid carrying weights over. This smaller experiment subset keeps the comparison practical on an Intel Mac. Each trial reports validation and test performance. The best configuration is selected by **validation accuracy**, not test accuracy.


In [ ]:
# A. Test hyperparameter: number of convolution filters.
import pandas as pd

experiment_rng = np.random.default_rng(2026)
experiment_indices = experiment_rng.choice(len(X_train_cnn), size=15000, replace=False)
X_exp = X_train_cnn[experiment_indices]
y_exp = y_train_onehot[experiment_indices]

# A new model is built and initialized for each trial.
def build_experiment_cnn(filters=32, dropout_rate=0.0):
    layers = [
        Input(shape=(28, 28, 1)),
        Conv2D(filters, 3, activation="relu"),
        MaxPooling2D(2),
        Conv2D(filters * 2, 3, activation="relu"),
        MaxPooling2D(2),
        Flatten(),
        Dense(64, activation="relu"),
    ]
    if dropout_rate > 0:
        layers.append(Dropout(dropout_rate))
    layers.append(Dense(10, activation="softmax"))
    candidate = Sequential(layers)
    candidate.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return candidate

experiment_results = []

def run_experiment(filters, dropout_rate):
    tf.keras.utils.set_random_seed(100)
    candidate = build_experiment_cnn(filters=filters, dropout_rate=dropout_rate)
    history = candidate.fit(
        X_exp, y_exp, epochs=3, batch_size=128,
        validation_split=0.2, verbose=2
    )
    test_loss, test_accuracy = candidate.evaluate(
        X_test_cnn, y_test_onehot, verbose=0
    )
    result = {
        "filters": filters,
        "dropout": dropout_rate,
        "val_accuracy": max(history.history["val_accuracy"]),
        "val_loss": min(history.history["val_loss"]),
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
    }
    experiment_results.append(result)
    print(f"Filters={filters}, dropout={dropout_rate}: "
          f"best val accuracy={result['val_accuracy']:.2%}, "
          f"test accuracy={test_accuracy:.2%}")
    return result

for num_filters in (16, 32, 64):
    run_experiment(filters=num_filters, dropout_rate=0.0)

print("\nFilter-only experiment results:")
display(pd.DataFrame(experiment_results).round(4))


In [ ]:
# B. Test presence/absence of regularization while holding filters constant.
# The no-dropout result was already recorded in Part A.
filter_trials = [r for r in experiment_results if r["dropout"] == 0]
best_plain = max(filter_trials, key=lambda r: r["val_accuracy"])
selected_filters = int(best_plain["filters"])
print("Filters chosen by validation accuracy:", selected_filters)

run_experiment(filters=selected_filters, dropout_rate=0.25)

results_df = pd.DataFrame(experiment_results)
results_df["configuration"] = results_df.apply(
    lambda r: f"{int(r['filters'])} filters / dropout {r['dropout']:.2f}", axis=1
)
print("\nAll controlled experiment results:")
display(results_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(len(results_df))
axes[0].bar(x - 0.18, results_df["val_accuracy"], width=0.36, label="Validation")
axes[0].bar(x + 0.18, results_df["test_accuracy"], width=0.36, label="Test")
axes[0].set_title("Classification accuracy")
axes[0].set_ylabel("Accuracy")
axes[0].set_xticks(x, results_df["configuration"], rotation=25, ha="right")
axes[0].legend()
axes[1].bar(x - 0.18, results_df["val_loss"], width=0.36, label="Validation")
axes[1].bar(x + 0.18, results_df["test_loss"], width=0.36, label="Test")
axes[1].set_title("Cross-entropy loss")
axes[1].set_ylabel("Loss")
axes[1].set_xticks(x, results_df["configuration"], rotation=25, ha="right")
axes[1].legend()
plt.tight_layout()
plt.show()

# Choose only from validation scores to avoid optimizing on the test set.
best_trial = max(experiment_results, key=lambda r: r["val_accuracy"])
best_filters = int(best_trial["filters"])
best_dropout = float(best_trial["dropout"])
print(f"Selected: filters={best_filters}, dropout={best_dropout:.2f}")
print(f"Validation accuracy for selected configuration: {best_trial['val_accuracy']:.2%}")
print(f"Its test accuracy (for reporting): {best_trial['test_accuracy']:.2%}")

no_dropout_result = next(r for r in experiment_results
                         if r["filters"] == selected_filters and r["dropout"] == 0.0)
dropout_result = next(r for r in experiment_results
                      if r["filters"] == selected_filters and r["dropout"] == 0.25)
print("Change in validation accuracy with dropout:",
      f"{(dropout_result['val_accuracy'] - no_dropout_result['val_accuracy'])*100:+.2f} percentage points")


### Reflection — Controlled experiments

Three convolution filter settings (**16, 32, 64**) were compared with identical training conditions. For the filter setting with the highest validation accuracy, I then compared **no dropout** against **25% dropout**. The table and two bar charts above report the actual validation/test accuracy and loss for every tested model, and print the winner and its settings automatically.

Changing the number of filters changes the network's representational capacity, but a larger network is not guaranteed to improve generalization. Dropout discourages over-reliance on individual hidden units, which may help with overfitting, although it can also reduce accuracy if the model is already underfitting. My final selection uses the **highest recorded validation accuracy** rather than choosing retrospectively from test results. Since experiments trained on a subset, I retrain the selected architecture on the **full training set** below before evaluating it as the final model.


# 5. Training Final Model and Evaluation

Build a freshly initialized network with the selected filter count and dropout rate, retrain on all 60,000 training images, and measure held-out test performance. The model is not reused from the hyperparameter experiments.


In [ ]:
tf.keras.utils.set_random_seed(2026)
final_model = build_experiment_cnn(filters=best_filters, dropout_rate=best_dropout)
final_model.summary()
final_history = final_model.fit(
    X_train_cnn, y_train_onehot,
    epochs=5, batch_size=128, validation_split=0.1,
    verbose=2
)
final_loss, final_accuracy = final_model.evaluate(
    X_test_cnn, y_test_onehot, verbose=0
)

print("\nHeld-out test performance:")
print(f"Baseline:  accuracy={baseline_accuracy:.2%}, loss={baseline_loss:.4f}")
print(f"CNN:       accuracy={cnn_accuracy:.2%}, loss={cnn_loss:.4f}")
print(f"Final:     accuracy={final_accuracy:.2%}, loss={final_loss:.4f}")
print(f"Final vs baseline: {(final_accuracy-baseline_accuracy)*100:+.2f} percentage points")
print(f"Final vs reference CNN: {(final_accuracy-cnn_accuracy)*100:+.2f} percentage points")

# Plot validation accuracy for the three full-data training runs.
plt.figure(figsize=(8, 4))
for label, history in [
    ("Linear baseline", baseline_history),
    ("Reference CNN", cnn_history),
    ("Selected final CNN", final_history)
]:
    plt.plot(history.history["val_accuracy"], label=label)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Full-training performance comparison")
plt.legend()
plt.tight_layout()
plt.show()

# Make the conclusions reflect the actual computed numbers.
if final_accuracy > cnn_accuracy:
    comparison = "The final model improved on the reference CNN."
elif final_accuracy < cnn_accuracy:
    comparison = "The final model did not outperform the reference CNN in this run."
else:
    comparison = "The final model matched the reference CNN in this run."
print("\nFinal reflection (based on the measured results):")
print(comparison)
print(f"Its test accuracy was {final_accuracy:.2%}, compared with "
      f"{baseline_accuracy:.2%} for the linear baseline and "
      f"{cnn_accuracy:.2%} for the reference CNN.")
print(f"The selected architecture had {best_filters} first-layer filters and "
      f"dropout rate {best_dropout:.2f}.")
print("Possible improvements: tune learning rate, train longer with early stopping, "
      "or add image augmentation. All should be selected on validation data.")


### Reflection — Final model

The final model uses the filter count and dropout setting selected by the controlled experiments, but starts with **new weights** and is trained on the full training set. Its actual test performance is displayed in the preceding cell beside the linear baseline and original CNN, with differences reported in **percentage points**.

The final accuracy reflects both the learned spatial features and choices such as network capacity, regularization, and training duration. A selected model is **not guaranteed** to beat the original CNN on every run, particularly when the experiment phase used only part of the training data. The printed, measured comparison is the appropriate conclusion for this run. With additional time, I would investigate learning-rate schedules, early stopping, data augmentation, and repeated random seeds, while choosing settings on validation data rather than on the test set.
